In [1]:
# ==========================================================
#  CLEAN stacked DDD Event Study (Industry-by-industry)
#  + Long-run control: dlogW (log min wage vs 2022 Jan base)
#  + Province & Date FE
#  + Clean control ("hole punching")
#  + Inference:
#       (1) Cluster-robust SE at province level (main)
#       (2) Wild Cluster Bootstrap (Rademacher, province-level)
#           - RESTRICTED (null-imposed) wild bootstrap
#           - Correct t-test for dlogW and Wald tests for JOINT/PRE/POST
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# -----------------------------
# 0. Parameters
# -----------------------------
B = 500
SEED = 42
np.random.seed(SEED)

base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
DATA = fr"{base}\14100355.csv"

L, R = 2, 3
BASE_K = -1

OUT = r"C:\Users\SC2zh\Desktop\S4 paper\results_wild_correct.csv"


def event_var(k: int) -> str:
    return f"event_{'m'+str(abs(k)) if k < 0 else 'p'+str(k)}_dose"


def build_wald_hypothesis(vars_):
    """
    Build a statsmodels-style linear restriction string:
        "x1 = 0, x2 = 0, x3 = 0"
    """
    if not vars_:
        return None
    return ", ".join([f"{v} = 0" for v in vars_])


def fit_clustered_ols(formula: str, data: pd.DataFrame, cluster_col: str):
    return smf.ols(formula, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": data[cluster_col]}
    )


def wild_cluster_bootstrap_restricted(
    data: pd.DataFrame,
    y_col: str,
    formula_unres: str,
    formula_res: str,
    cluster_col: str,
    stat_type: str,
    target=None,
    B: int = 999,
    seed: int = 42
):
    """
    Correct restricted wild cluster bootstrap.

    Inputs:
      - formula_unres: unrestricted model (full)
      - formula_res: restricted model under H0 (drop tested regressors)
      - stat_type:
          "t"    -> target must be a variable name (single coefficient t-stat)
          "wald" -> target must be a restriction string like "a=0, b=0"
      - cluster_col: province

    Returns:
      dict with:
        orig_stat, boot_pvalue, boot_stats (np.array)
    """
    rng = np.random.default_rng(seed)

    # 1) Original statistic from UNRESTRICTED model (clustered)
    m_unres = fit_clustered_ols(formula_unres, data, cluster_col)

    if stat_type == "t":
        v = target
        if v not in m_unres.params.index:
            return {"orig_stat": np.nan, "boot_pvalue": np.nan, "boot_stats": np.array([])}
        orig_stat = float(np.abs(m_unres.tvalues[v]))

    elif stat_type == "wald":
        hyp = target
        if hyp is None:
            return {"orig_stat": np.nan, "boot_pvalue": np.nan, "boot_stats": np.array([])}
        # scalar=False keeps matrix; but we want a scalar stat
        w = m_unres.wald_test(hyp, scalar=True)
        orig_stat = float(w.statistic)

    else:
        raise ValueError("stat_type must be 't' or 'wald'")

    # 2) Fit RESTRICTED model under H0 (clustered)
    m_res = fit_clustered_ols(formula_res, data, cluster_col)
    yhat0 = m_res.fittedvalues.to_numpy()
    uhat0 = m_res.resid.to_numpy()

    clusters = data[cluster_col].to_numpy()
    uniq = np.unique(clusters)

    # Pre-allocate
    boot_stats = np.empty(B, dtype=float)

    # 3) Bootstrap loop: y* = yhat0 + uhat0 * w_g
    for b in range(B):
        # Rademacher weights at cluster level
        w_map = {g: rng.choice([-1, 1]) for g in uniq}
        w = np.array([w_map[g] for g in clusters], dtype=float)

        y_star = yhat0 + uhat0 * w

        # Put y* into a copy (avoid modifying original)
        dstar = data.copy()
        dstar["_y_star"] = y_star

        # Refit UNRESTRICTED on y* (clustered) and compute SAME statistic
        m_star = fit_clustered_ols(
            formula_unres.replace(y_col, "_y_star"),
            dstar,
            cluster_col
        )

        if stat_type == "t":
            boot_stats[b] = float(np.abs(m_star.tvalues.get(target, np.nan)))
        else:
            try:
                boot_stats[b] = float(m_star.wald_test(target, scalar=True).statistic)
            except Exception:
                boot_stats[b] = np.nan

    boot_stats = boot_stats[np.isfinite(boot_stats)]
    if boot_stats.size == 0 or not np.isfinite(orig_stat):
        return {"orig_stat": orig_stat, "boot_pvalue": np.nan, "boot_stats": boot_stats}

    # 4) p-value = share of bootstrap stats >= original stat
    pval = float(np.mean(boot_stats >= orig_stat))
    return {"orig_stat": orig_stat, "boot_pvalue": pval, "boot_stats": boot_stats}


# -----------------------------
# 1. Read and clean data
# -----------------------------
raw = pd.read_csv(DATA)

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"])

raw["province"] = raw["province"].astype(str).str.strip()
raw["industry"] = raw["industry"].astype(str).str.strip()

raw["min_wage"] = pd.to_numeric(raw["min_wage"], errors="coerce")
raw["employment_rate"] = pd.to_numeric(raw["employment_rate"], errors="coerce")

raw["m_id"] = raw["date"].dt.year * 12 + raw["date"].dt.month

# -----------------------------
# 2. Construct 2022 base wage (first obs in 2022 per province)
# -----------------------------
policy = raw.dropna(subset=["min_wage"]).copy()
policy = policy.sort_values(["province", "date"])

base_wage = (
    policy.loc[policy["date"].dt.year == 2022, ["province", "date", "min_wage"]]
    .sort_values(["province", "date"])
    .groupby("province", as_index=False)
    .head(1)
    .rename(columns={"min_wage": "wage_base_2022"})
    [["province", "wage_base_2022"]]
)

raw = raw.merge(base_wage, on="province", how="left")
raw["dlogW"] = np.log(raw["min_wage"]) - np.log(raw["wage_base_2022"])

# -----------------------------
# 3. Regression-ready panel
# -----------------------------
df = raw.dropna(subset=["employment_rate", "min_wage", "dlogW"]).copy()
df = df.sort_values(["province", "date"])

# -----------------------------
# 4. Identify 2023+ events
# -----------------------------
df["dW"] = df.groupby("province")["min_wage"].diff()
df["lagW"] = df.groupby("province")["min_wage"].shift(1)
df["dose"] = (df["dW"] / df["lagW"]) * 100

events = (
    df.loc[(df["dose"] > 0) & (df["date"] >= "2023-01-01"),
           ["province", "date", "dose", "m_id"]]
    .drop_duplicates()
    .rename(columns={"date": "t0", "m_id": "m0"})
    .reset_index(drop=True)
)

print(f"Identified 2023+ MW increase events: {len(events)}")

# -----------------------------
# 5. Global hole-punching: flag any obs in any event window (within province)
# -----------------------------
df["in_any_window"] = 0
for prov, g in events.groupby("province"):
    m0_list = g["m0"].tolist()
    idx = df["province"].eq(prov)
    m = df.loc[idx, "m_id"].values
    hit = np.zeros_like(m, dtype=bool)
    for m0 in m0_list:
        hit |= (m >= m0 - L) & (m <= m0 + R)
    df.loc[idx, "in_any_window"] = hit.astype(int)

# -----------------------------
# 6. Industry-by-industry estimation + correct wild bootstrap inference
# -----------------------------
industries = sorted(df["industry"].unique())
all_results = []

for ind in industries:
    dfi = df[df["industry"] == ind].copy()
    stack_list = []

    for ev in events.itertuples(index=False):
        ev_prov, m0, dose = ev.province, ev.m0, ev.dose

        treated = dfi[dfi["province"] == ev_prov].copy()
        treated["EventTime"] = treated["m_id"] - m0
        treated = treated[(treated["EventTime"] >= -L) & (treated["EventTime"] <= R)]

        # Event-time × dose regressors for treated
        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            treated[event_var(k)] = (treated["EventTime"] == k).astype(int) * dose

        if treated.empty:
            continue

        # Clean controls: other provinces not in any window at all
        controls = dfi[(dfi["province"] != ev_prov) & (dfi["in_any_window"] == 0)].copy()

        # Set event regressors to zero for controls
        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            controls[event_var(k)] = 0.0

        stack_list.append(pd.concat([treated, controls], ignore_index=True))

    if not stack_list:
        continue

    stack = pd.concat(stack_list, ignore_index=True)

    # Identify valid event regressors
    event_cols = [c for c in stack.columns if c.startswith("event_")]
    valid_events = [c for c in event_cols if stack[c].sum() != 0]

    if not valid_events:
        continue

    # UNRESTRICTED formula
    formula_unres = (
        "employment_rate ~ dlogW + "
        + " + ".join(valid_events)
        + " + C(province) + C(date)"
    )

    # --- Clustered main model (for reporting baseline coef/p) ---
    m = fit_clustered_ols(formula_unres, stack, "province")

    # Event group lists
    pre_events = [v for v in valid_events if "_m" in v]
    post_events = [v for v in valid_events if "_p" in v]

    # Restricted formulas under each H0:
    # (a) H0: dlogW = 0  => drop dlogW from RHS
    formula_res_long = (
        "employment_rate ~ "
        + " + ".join(valid_events)
        + " + C(province) + C(date)"
    )

    # (b) H0: all event coefficients = 0 => drop all event vars
    formula_res_joint = (
        "employment_rate ~ dlogW + C(province) + C(date)"
    )

    # (c) H0: PRE-only = 0 => drop pre vars only (keep post)
    keep_for_pre_null = [v for v in valid_events if v not in pre_events]
    formula_res_pre = (
        "employment_rate ~ dlogW"
        + ((" + " + " + ".join(keep_for_pre_null)) if keep_for_pre_null else "")
        + " + C(province) + C(date)"
    )

    # (d) H0: POST-only = 0 => drop post vars only (keep pre)
    keep_for_post_null = [v for v in valid_events if v not in post_events]
    formula_res_post = (
        "employment_rate ~ dlogW"
        + ((" + " + " + ".join(keep_for_post_null)) if keep_for_post_null else "")
        + " + C(province) + C(date)"
    )

    # Hypothesis strings for Wald tests (UNRESTRICTED)
    hyp_joint = build_wald_hypothesis(valid_events)
    hyp_pre = build_wald_hypothesis(pre_events)
    hyp_post = build_wald_hypothesis(post_events)

    # --- Correct restricted wild cluster bootstrap p-values ---
    # Long (t-test)
    boot_long = wild_cluster_bootstrap_restricted(
        data=stack,
        y_col="employment_rate",
        formula_unres=formula_unres,
        formula_res=formula_res_long,
        cluster_col="province",
        stat_type="t",
        target="dlogW",
        B=B,
        seed=SEED
    )

    # JOINT (Wald)
    boot_joint = wild_cluster_bootstrap_restricted(
        data=stack,
        y_col="employment_rate",
        formula_unres=formula_unres,
        formula_res=formula_res_joint,
        cluster_col="province",
        stat_type="wald",
        target=hyp_joint,
        B=B,
        seed=SEED + 1
    )

    # PRE (Wald) if any
    if pre_events:
        boot_pre = wild_cluster_bootstrap_restricted(
            data=stack,
            y_col="employment_rate",
            formula_unres=formula_unres,
            formula_res=formula_res_pre,
            cluster_col="province",
            stat_type="wald",
            target=hyp_pre,
            B=B,
            seed=SEED + 2
        )
    else:
        boot_pre = {"orig_stat": np.nan, "boot_pvalue": np.nan}

    # POST (Wald) if any
    if post_events:
        boot_post = wild_cluster_bootstrap_restricted(
            data=stack,
            y_col="employment_rate",
            formula_unres=formula_unres,
            formula_res=formula_res_post,
            cluster_col="province",
            stat_type="wald",
            target=hyp_post,
            B=B,
            seed=SEED + 3
        )
    else:
        boot_post = {"orig_stat": np.nan, "boot_pvalue": np.nan}

    # -----------------------------
    # Store: baseline + bootstrap p-values
    # -----------------------------
    all_results.append({
        "industry": ind,
        "term": "dlogW",
        "coef": float(m.params.get("dlogW", np.nan)),
        "cluster_p": float(m.pvalues.get("dlogW", np.nan)),
        "wild_stat": boot_long["orig_stat"],
        "wild_p": boot_long["boot_pvalue"],
        "test": "t(|t|)"
    })

    all_results.append({
        "industry": ind,
        "term": "JOINT_FULL",
        "coef": np.nan,
        "cluster_p": float(m.wald_test(hyp_joint, scalar=True).pvalue) if hyp_joint else np.nan,
        "wild_stat": boot_joint["orig_stat"],
        "wild_p": boot_joint["boot_pvalue"],
        "test": "Wald"
    })

    all_results.append({
        "industry": ind,
        "term": "JOINT_PRE",
        "coef": np.nan,
        "cluster_p": float(m.wald_test(hyp_pre, scalar=True).pvalue) if hyp_pre else np.nan,
        "wild_stat": boot_pre["orig_stat"],
        "wild_p": boot_pre["boot_pvalue"],
        "test": "Wald"
    })

    all_results.append({
        "industry": ind,
        "term": "JOINT_POST",
        "coef": np.nan,
        "cluster_p": float(m.wald_test(hyp_post, scalar=True).pvalue) if hyp_post else np.nan,
        "wild_stat": boot_post["orig_stat"],
        "wild_p": boot_post["boot_pvalue"],
        "test": "Wald"
    })

    print(f"[DONE] {ind}  |  dlogW(wild p)={boot_long['boot_pvalue']:.4f}  "
          f"JOINT(wild p)={boot_joint['boot_pvalue']:.4f}")

# -----------------------------
# 7. Output
# -----------------------------
res_df = pd.DataFrame(all_results)
res_df.to_csv(OUT, index=False)

print("\n=== Correct Wild Cluster Bootstrap Results (Saved) ===")
print(res_df.head(30).to_string(index=False))
print(f"\nSaved to: {OUT}")


Identified 2023+ MW increase events: 23
[DONE] Accommodation and food services  |  dlogW(wild p)=0.0420  JOINT(wild p)=0.0380
[DONE] Agriculture  |  dlogW(wild p)=0.4280  JOINT(wild p)=0.1980
[DONE] Business, building and other support services  |  dlogW(wild p)=0.6060  JOINT(wild p)=0.7840
[DONE] Construction  |  dlogW(wild p)=0.9440  JOINT(wild p)=0.9720
[DONE] Educational services  |  dlogW(wild p)=0.9500  JOINT(wild p)=0.2020
[DONE] Finance, insurance, real estate, rental and leasing  |  dlogW(wild p)=0.6540  JOINT(wild p)=0.6240
[DONE] Forestry, fishing, mining, quarrying, oil and gas  |  dlogW(wild p)=0.7320  JOINT(wild p)=0.7060
[DONE] Goods-producing sector  |  dlogW(wild p)=0.8360  JOINT(wild p)=0.4020
[DONE] Health care and social assistance  |  dlogW(wild p)=0.6860  JOINT(wild p)=0.1800
[DONE] Information, culture and recreation  |  dlogW(wild p)=0.8820  JOINT(wild p)=0.4340
[DONE] Manufacturing  |  dlogW(wild p)=0.3640  JOINT(wild p)=0.1760
[DONE] Other services (except pub